In [1]:
############################################################################################
# STEP3_VERIFY_COSTS_TIMES.py                                                              # 
# kcazzato 09/05/2025 updates                                                              #
# ----- Translated SAS processsing                                                         #
# Craig Heither, rev. 05-06-2016                                                           #
# 03-09-2017: revised to include port summaries for Atlanta, Las Vegas and Cincinnati. 	   #
# 08-30-2017: revised to allow intrazonal inland waterways movements for non-CMAP zones.   #         
#                                                                                          #
# This program                                                                             #
#  - verifies all appropriate modes are available between zone pairs.                      #
#  - creates files to verify that the "best" domestic port logic is reasonable             #                                                                             # 
#                                                                                          #
#-- Input files needed (located in ..\input_data\post_processing\):                #
#	  -            #

#                                                                                          #
#
#-- Output (../output_data/post_processing/)	                                #
#	  - port_detail_review_YYYY.csv      #
#	  - port_summary_review__YYYY.csv      #

#                                                                                          #
############################################################################################

# ---------------------------------------------------------------
# Import Directories
# ---------------------------------------------------------------
import sys, os, shutil, math, yaml, io, random
import time
import pandas as pd, numpy as np

In [2]:
# ---------------------------------------------------------------
# Define paths
# ---------------------------------------------------------------
##-- System inputs
year = '2022'       ##-- Model run year, used to label output files
flag140 = 0         ##-- Flag if logistics node 140 is active (=1) or not (=0)
flag2022 = 0        ##-- Flag if year ==2022, if so (=0) then node 143 is not active, otherwise it is (=1)

##-- File directories
programDir = 'S:/AdminGroups/ResearchAnalysis/kcc/FY26/FreightSkims/TranslateSAS/2_dev/Meso_Freight_Skim_Setup_c25q2_2022/Database/post_processing'  ##-- Database/post_processing
databaseDir = os.path.abspath(os.path.join(programDir, "../"))             ##-- Database     
inDir = os.path.join(databaseDir + '/input_data/post_processing')          ##-- Database/input_data/post_processing
outDir = os.path.join(databaseDir + '/output_data/post_processing')        ##-- Database/output_data/post_processing

##-- Inputs 
pth_nodeznmeso= os.path.join(inDir + "/node_zone_meso.yaml")           ##-- Zone and node ranges by mode and region (CMAP, logistics, non-CMAP)
pth_modepathcosts = os.path.join(outDir + "/data_all_modepath_costs_" + year + ".csv")
##-- Outputs
outPortDetail= os.path.join(outDir + "/port_detail_review_" + year + ".csv")     ##-- Outpath for port detail file
outPortSummary= os.path.join(outDir + "/port_summary_review_" + year + ".csv")   ##-- Outpath for port summary file


In [3]:
# ---------------------------------------------------------------
# Import Data
# ---------------------------------------------------------------
in_modepathcosts = pd.read_csv(pth_modepathcosts)
with open(pth_nodeznmeso, 'r') as file:         ##-- Zone and node ranges by mode and region (CMAP, logistics, non-CMAP)
    in_nodeznmeso = yaml.safe_load(file)

C:\Users\kcazzato\AppData\Local\Temp\ipykernel_36700\3985797046.py:4: DtypeWarning: Columns (201,206,261,262,263,264,267) have mixed types. Specify dtype option on import or set low_memory=False.
  in_modepathcosts = pd.read_csv(pth_modepathcosts)


In [4]:
# Modepaths
# Inland Water: 1
# Carload direct/indirect: 3, 4
# IMX direct/indirect: 13, 14
# FTL direct: 31
# FTL indirect: 32
# LTL direct: 46
# LTL indirect: 39
# Air: 47
allModepaths = list(range(1,51))

In [6]:
# ---------------------------------------------------------------
# Check non-CMAP U.S. intrazonal movements
# ---------------------------------------------------------------
cond_nonCMAPIntra = (((in_modepathcosts['origin'] <= in_nodeznmeso['LEZ']) & (in_modepathcosts['origin'] >= in_nodeznmeso['FEZ'])) & 
                     (in_modepathcosts['destination'] == in_modepathcosts['origin']))
qc1 = in_modepathcosts.loc[cond_nonCMAPIntra].copy()

# Carload direct/indirect: 3, 4; IMX direct/indirect: 13, 14; FTL direct: 31; FTL indirect: 32; LTL direct: 46; LTL indirect: 39; Air: 47
modepaths1 = [3, 4, 13, 14, 31, 32, 39, 46, 47]
for _, row in qc1.iterrows(): 
    for path in modepaths1:
        if pd.isnull(row[f'time{path}']):
            print("Missing Time Data for non-CMAP U.S. intrazonal movements: ")
            print(row[['origin', 'destination', f'time{path}']])
            sys.exit()
        if pd.isnull(row[f'cost{path}']):
            print("Missing Cost Data for non-CMAP U.S. intrazonal movements: ")
            print(row[['origin', 'destination', f'cost{path}']])
            sys.exit()
            
# Check remaining mode paths for no data, do not expect any data in these
modepaths2 = [2, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 48, 49, 50, 51, 52, 53, 54]
for _, row in qc1.iterrows(): 
    for path in modepaths2:
        if row[f'time{path}'] > 0:
            print("Bad Time Data for non-CMAP U.S. intrazonal movements: ")
            print(row[['origin', 'destination', f'time{path}']])
            sys.exit()
        if row[f'cost{path}'] > 0:
            print("Bad Cost Data for non-CMAP U.S. intrazonal movements: ")
            print(row[['origin', 'destination', f'cost{path}']])
            sys.exit()

In [10]:
# ---------------------------------------------------------------
# Check Canada to CMAP
# ---------------------------------------------------------------
cond_CanadaCMAP = ((in_modepathcosts['origin'] == in_nodeznmeso['Canada'])& 
                     (in_modepathcosts['destination'] <= in_nodeznmeso['LIZ']))
qc1 = in_modepathcosts.loc[cond_CanadaCMAP].copy()

# water ports [1,2], FTL direct [31], LTL direct [46], air [47-50]
modepaths1 = [1, 2, 31, 46, 47, 48, 50]
if flag2022!=0:
    modepaths1.append(49)
for _, row in qc1.iterrows(): 
    for path in modepaths1:
        if pd.isnull(row[f'time{path}']):
            print("Missing Time Data for non-CMAP U.S. intrazonal movements: ")
            print(row)
            sys.exit()
        if pd.isnull(row[f'cost{path}']):
            print("Missing Cost Data for non-CMAP U.S. intrazonal movements: ")
            print(row)
            sys.exit()
            
# Check International Water - ensure no data
modepaths1 = [51, 52, 53, 54]
for _, row in qc1.iterrows(): 
    for path in modepaths1:
        if row[f'time{path}'] > 0:
            print("Bad Time (international water) Data for Canada to CMAP movements: ")
            print(row)
            sys.exit()
        if row[f'cost{path}'] > 0:
            print("Bad Cost (international water) Data for Canada to CMAP movements: ")
            print(row)
            sys.exit()

# Check some trail is available for certain zone pairs
# time and costs 3-30
for _, row in qc1.iterrows(): 
    idx = 3
    while idx <= 30:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: No Rail Service Time found for Canada to CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: No Rail Service Cost found for Canada to CMAP movements: ")
            print(row)
            sys.exit()        

ERROR: No Rail Service Time found for Canada to CMAP movements: 
Shpmile        NaN
origin         310
destination      1
cFTL40dir      NaN
tFTL40dir      NaN
              ... 
cost54         NaN
time54         NaN
mile54         NaN
LHmile54       NaN
DRmile54       NaN
Name: 129032, Length: 563, dtype: object


C:\Users\kcazzato\AppData\Local\Temp\ipykernel_36700\1620982290.py:41: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  if ~(row[f'time{idx}'] > 0):


SystemExit: 

c:\Users\kcazzato\.conda\envs\anacondaINSTALL\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ---------------------------------------------------------------
# Check Canada to non-CMAP U.S.
# ---------------------------------------------------------------
cond_CanadaNonCMAP = ((in_modepathcosts['Origin'] == in_nodeznmeso['Canada'])& 
                     ((in_modepathcosts['destination'] <= in_nodeznmeso['LEZ']) & (in_modepathcosts['destination'] >= in_nodeznmeso['FEZ'])))
qc1 = in_modepathcosts.loc[cond_CanadaNonCMAP].copy()

In [ ]:
# PART 1: Exclude Hawaii
qc12 = qc1.loc[(qc1['destination'] != in_nodeznmeso['Hawaii1']) & (qc1['destination'] != in_nodeznmeso['Hawaii2'])].copy()

# Minimum Options available to all zones should be: FTL direct [31], LTL direct [46], air [47], international water [51-54]
modepaths1 = [31, 46, 47, 51, 52, 53, 54]
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data for Canada to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data for Canada to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()

# No CMAP water [2 only] or air [48-50] should be available
modepaths1 = [2, 48, 50]
if flag2022!=0:
    modepaths1.append(49)
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if row[f'time{path}'] > 0:
            print("Bad Time (CMAP air or CMAP water) Data for Canada to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if row[f'cost{path}'] > 0:
            print("Bad Cost (CMAP air or CMAP water) Data for Canada to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()

# verify at least some rail is available for certain zone pairs
# time and costs 3-30
# Exclude Alaska
qc12 = qc12.loc[(qc12['destination'] != in_nodeznmeso['Alaska'])].copy()
for _, row in qc12.iterrows(): 
    idx = 3
    for while idx <= 30:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: No Rail Service found for Canada to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: No Rail Service found for Canada to non-CMAP U.S. movements: ")
            print(row)
            sys.exit() 
            
# To Alaska Only: but there should be NO rail service between Canada and Alaska
qc12 = qc1.loc[(qc1['destination'] == in_nodeznmeso['Alaska'])].copy()
for _, row in qc12.iterrows(): 
    idx = 3
    for while idx <= 30:
        if row[f'time{idx}'] > 0:
            print("ERROR: There should be NO Rail Service Time between Canada & Alaska: ")
            print(row)
            sys.exit()
        if row[f'cost{idx}'] > 0:
            print("ERROR: There should be NO Rail Service Cost between Canada & Alaska: ")
            print(row)
            sys.exit() 

In [ ]:
# PART 2: Canada to Hawaii
qc12 = qc1.loc[(qc1['destination'] == in_nodeznmeso['Hawaii1']) | (qc1['destination'] == in_nodeznmeso['Hawaii2'])].copy()

# ONLY air [47] and international water [51-54] should be available
modepaths1 = [47, 51, 52, 53]
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data (air or international shipping) for Canada to Hawaii movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data (air or international shipping) for Canada to Hawaii movements: ")
            print(row)
            sys.exit()

# other modes must NOT be available
for _, row in qc12.iterrows(): 
    idx = 1
    for while idx <= 46:
        if row[f'time{idx}'] > 0:
            print("Bad Time (unavailable modes) Data for Canada to Hawaii movements: ")
            print(row)
            sys.exit()
        if row[f'cost{idx}'] > 0:
            print("Bad Cost (unavailable modes) Data for Canada to Hawaii movements: ")
            print(row)
            sys.exit() 
            
    idx = 48
    for while idx <= 50:
        if row[f'time{idx}'] > 0:
            print("Bad Time (unavailable modes) Data for Canada to Hawaii movements: ")
            print(row)
            sys.exit()
        if row[f'cost{idx}'] > 0:
            print("Bad Cost (unavailable modes) Data for Canada to Hawaii movements: ")
            print(row)
            sys.exit() 

In [ ]:
# ---------------------------------------------------------------
# Check Mexico to CMAP
# ---------------------------------------------------------------
cond_MexicoCMAP = ((in_modepathcosts['Origin'] == in_nodeznmeso['Mexico'])& 
                     (in_modepathcosts['destination'] <= in_nodeznmeso['LIZ']))
qc1 = in_modepathcosts.loc[cond_MexicoCMAP].copy()

# Minimum Options available to all zones should be: water ports [1,2], FTL direct [31], LTL direct [46], air [47-50]
modepaths1 = [1, 2, 31, 46, 47, 48, 50]
if flag2022!=0:
    modepaths1.append(49)
for _, row in qc1.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time/Cost Data for Mexico to CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Time/Cost Data for Mexico to CMAP movements: ")
            print(row)
            sys.exit()
            
# No international water [51-54] should be available
modepaths1 = [51, 52, 53, 54]
for _, row in qc1.iterrows(): 
    for path in modepaths1:
        if row[f'time{path}'] > 0:
            print("Bad Time (international water) Data for Mexico to CMAP movements: ")
            print(row)
            sys.exit()
        if row[f'cost{path}'] > 0:
            print("Bad Cost (international water) Data for Mexico to CMAP movements: ")
            print(row)
            sys.exit()
            
# verify at least some rail is available for certain zone pairs
for _, row in qc12.iterrows(): 
    idx = 3
    for while idx <= 30:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: No Rail Service Time found for Mexico to CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: No Rail Service Cost found for Mexico to CMAP movements: ")
            print(row)
            sys.exit() 

In [ ]:
# ---------------------------------------------------------------
# Check Mexico to non-CMAP U.S.
# ---------------------------------------------------------------
cond_MexicoNonCMAP = ((in_modepathcosts['Origin'] == in_nodeznmeso['Mexico'])& 
                     ((in_modepathcosts['destination'] <= in_nodeznmeso['LEZ']) & (in_modepathcosts['destination'] >= in_nodeznmeso['FEZ'])))
qc1 = in_modepathcosts.loc[cond_MexicoNonCMAP].copy()

# Exclude Hawaii
qc12 = qc1.loc[(qc1['destination'] != in_nodeznmeso['Hawaii1']) & (qc1['destination'] != in_nodeznmeso['Hawaii2'])].copy()

# Minimum Options available to all zones should be: FTL direct [31], LTL direct [46], air [47], international water [51-54]
modepaths1 = [31, 46, 47, 51, 52, 53, 54]
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data for Mexico to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data for Mexico to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()

# No CMAP water [2 only] or air [48-50] should be available
modepaths1 = [2, 48, 50]
if flag2022!=0:
    modepaths1.append(49)
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if row[f'time{path}'] > 0:
            print("Bad Time (CMAP air or CMAP water) Data for Mexico to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if row[f'cost{path}'] > 0:
            print("Bad Cost (CMAP air or CMAP water) Data for Mexico to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()

# verify at least some rail is available for certain zone pairs
# time and costs 3-30
# Exclude Alaska
qc12 = qc12.loc[(qc12['destination'] != in_nodeznmeso['Alaska'])].copy()
for _, row in qc12.iterrows(): 
    idx = 3
    for while idx <= 30:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: No Rail Service found for Mexico to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: No Rail Service found for Mexico to non-CMAP U.S. movements: ")
            print(row)
            sys.exit() 
            
# To Alaska Only: but there should be NO rail service between Canada and Alaska
qc12 = qc1.loc[(qc1['destination'] == in_nodeznmeso['Alaska'])].copy()
for _, row in qc12.iterrows(): 
    idx = 3
    for while idx <= 30:
        if row[f'time{idx}'] > 0:
            print("ERROR: There should be NO Rail Service Time between Mexico & Alaska: ")
            print(row)
            sys.exit()
        if row[f'cost{idx}'] > 0:
            print("ERROR: There should be NO Rail Service Cost between Mexico & Alaska: ")
            print(row)
            sys.exit() 

In [ ]:
# ---------------------------------------------------------------
# Check Hawaii to everywhere except Hawaii
# ---------------------------------------------------------------
cond_HawaiinotHawaii = (((in_modepathcosts['Origin'] == in_nodeznmeso['Hawaii1']) | (in_modepathcosts['Origin'] == in_nodeznmeso['Hawaii2']))& 
                     ((in_modepathcosts['destination'] != in_nodeznmeso['Hawaii1']) & (in_modepathcosts['destination'] != in_nodeznmeso['Hawaii2'])))
qc1 = in_modepathcosts.loc[cond_HawaiinotHawaii].copy()

In [ ]:
# PART 1. CMAP zones
qc12 = qc1.loc[(qc1['destination'] <= in_nodeznmeso['LIZ'])].copy()

# ONLY air [47-50] and international water [51-54] should be available to CMAP zones
modepaths1 = [47, 48, 50, 51, 52, 53, 54]
if flag2022!=0:
    modepaths1.append(49)
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data (air or international shipping) for Hawaii to CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data (air or international shipping) for Hawaii to CMAP movements: ")
            print(row)
            sys.exit()

# ONLY air [47-50] and international water [51-54] should be available to CMAP zones
for _, row in qc12.iterrows(): 
    idx = 1
    for while idx <= 46:
        if row[f'time{idx}'] > 0:
            print("ERROR: Bad Time for Hawaii to CMAP movements: ")
            print(row)
            sys.exit()
        if row[f'cost{idx}'] > 0:
            print("ERROR: Bad Cost for Hawaii to CMAP movements: ")
            print(row)
            sys.exit() 

In [ ]:
# PART 2. Non-CMAP zones
qc12 = qc1.loc[(qc1['destination'] > in_nodeznmeso['LIZ'])].copy()

# ONLY air [47] and international water [51-54] should be available to non-CMAP zone
modepaths1 = [47, 51, 52, 53, 54]
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data (air or international shipping) for Hawaii to non-CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data (air or international shipping) for Hawaii to non-CMAP movements: ")
            print(row)
            sys.exit()

# ONLY air [47] and international water [51-54] should be available to non-CMAP zones
for _, row in qc12.iterrows(): 
    idx = 1
    for while idx <= 46:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: Bad Time for Hawaii to non-CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: Bad Cost for Hawaii to non-CMAP movements: ")
            print(row)
            sys.exit() 
    idx = 48
    for while idx <= 50:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: Bad Time for Hawaii to non-CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: Bad Cost for Hawaii to non-CMAP movements: ")
            print(row)
            sys.exit() 

In [ ]:
# ---------------------------------------------------------------
# Check Alaska to everywhere except Alaska
# ---------------------------------------------------------------
cond_AlaskanotAlaska = ((in_modepathcosts['Origin'] == in_nodeznmeso['Alaska']) & 
                     (in_modepathcosts['destination'] != in_nodeznmeso['Alaska']))
qc1 = in_modepathcosts.loc[cond_AlaskanotAlaska].copy()

In [ ]:
# PART 1. CMAP zones
qc12 = qc1.loc[(qc1['destination'] <= in_nodeznmeso['LIZ'])].copy()

# Minimum Options available to all zones should be: FTL direct [31], LTL direct [46], air [47-50], international water [51-54]
modepaths1 = [31, 46, 47, 48, 50, 51, 52, 53, 54]
if flag2022!=0:
    modepaths1.append(49)
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data (truck, air or international shipping) for Alaska to CMAP movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data (truck, air or international shipping) for Alaska to CMAP movements: ")
            print(row)
            sys.exit()

# No CMAP water [1-2] or rail [3-30] should be available
for _, row in qc12.iterrows(): 
    idx = 1
    for while idx <= 30:
        if row[f'time{idx}'] > 0:
            print("ERROR: Bad Time (CMAP water or rail) Data for Alaska to CMAP movements: ")
            print(row)
            sys.exit()
        if row[f'cost{idx}'] > 0:
            print("ERROR: Bad Cost (CMAP water or rail) Data for Alaska to CMAP movements: ")
            print(row)
            sys.exit() 

In [ ]:
# PART 2. Non-CMAP zones
cond_Alaska2 = (((qc1['destination'] > in_nodeznmeso['LIZ']) & (qc1['destination'] <= in_nodeznmeso['LEZ'])) & 
                     ((in_modepathcosts['destination'] != in_nodeznmeso['Hawaii1']) & (in_modepathcosts['destination'] != in_nodeznmeso['Hawaii2'])))
qc12 = qc1.loc[cond_Alaska2].copy()


# Minimum Options available to all zones should be: FTL direct [31], LTL direct [46], air [47], international water [51-54]
modepaths1 = [31, 46, 47, 51, 52, 53, 54]
for _, row in qc12.iterrows(): 
    for path in modepaths1:
        if ~(row[f'time{path}'] > 0):
            print("Missing Time Data (truck, air or international shipping) for Alaska to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{path}'] > 0):
            print("Missing Cost Data (truck, air or international shipping) for Alaska to non-CMAP U.S. movements: ")
            print(row)
            sys.exit()

# ONLY air [47] and international water [51-54] should be available to non-CMAP zones
for _, row in qc12.iterrows(): 
    idx = 1
    for while idx <= 46:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: Bad Time (CMAP water, rail or air) Data for Alaska to non-CMAP U.S. movement: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: Bad Cost (CMAP water, rail or air) Data for Alaska to non-CMAP U.S. movement: ")
            print(row)
            sys.exit() 
    idx = 48
    for while idx <= 50:
        if ~(row[f'time{idx}'] > 0):
            print("ERROR: Bad Time (CMAP water, rail or air) Data for Alaska to non-CMAP U.S. movement: ")
            print(row)
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            print("ERROR: Bad Cost (CMAP water, rail or air) Data for Alaska to non-CMAP U.S. movement: ")
            print(row)
            sys.exit() 

In [ ]:
# PART 3: Foreign zones (except Canada/Mexico and Hawaii)


In [ ]:
# ---------------------------------------------------------------
# Check U.S. to Foreign
# ---------------------------------------------------------------

In [ ]:
# ---------------------------------------------------------------
# Check non-CMAP U.S. to non-CMAP U.S. (excluding Alaska/Hawaii)
# ---------------------------------------------------------------

In [ ]:
# ---------------------------------------------------------------
# Verify Every Pair has at least One Viable Option
# ---------------------------------------------------------------

In [ ]:
# ---------------------------------------------------------------
# Verify Mode Options for DIRECT Shipments
# ---------------------------------------------------------------

In [ ]:
# ---------------------------------------------------------------
# Verify Mode Options for INDIRECT Shipments
# ---------------------------------------------------------------

In [ ]:
# -------------------------------------------------------------------------------------
# Review specific movements to verify which domestic port is used for foreign trade 
# -------------------------------------------------------------------------------------
# Kansas City, Chicago, Denver, Atlanta, Las Vegas, Cincinnati
# to Eastern Asia
# to Europe
# to Rest of Americas

In [ ]:
# Define Conditions
cond_nonCMAPIntra = (((in_modepathcosts['origin'] <= in_nodeznmeso['LEZ']) & (in_modepathcosts['origin'] >= in_nodeznmeso['FEZ'])) & 
                     (in_modepathcosts['destination'] == in_modepathcosts['origin']))

cond_CanadaCMAP = ((in_modepathcosts['Origin'] == in_nodeznmeso['Canada'])& 
                     (in_modepathcosts['destination'] <= in_nodeznmeso['LIZ']))

cond_CanadaNonCMAP = ((in_modepathcosts['Origin'] == in_nodeznmeso['Canada'])& 
                     ((in_modepathcosts['destination'] <= in_nodeznmeso['LEZ']) & (in_modepathcosts['destination'] >= in_nodeznmeso['FEZ'])))\

# Sub-conditions
cond_noHawaii = ((qcDataset['destination'] != in_nodeznmeso['Hawaii1']) & (qcDataset['destination'] != in_nodeznmeso['Hawaii2']))
cond_noAlaska = (qcDataset['destination'] != in_nodeznmeso['Alaska'])
cond_yesAlaska = (qc1['destination'] == in_nodeznmeso['Alaska'])

In [ ]:
# Define modepath combinations for 'Missing Data' checks
# non-CMAP U.S. intrazonal movements
# Canada to CMAP movements
# Canada to non-CMAP U.S. movements
# (air or international shipping) for Canada to Hawaii movements
# Mexico to CMAP movements
# Mexico to non-CMAP U.S. movements
# (air or international shipping) for Hawaii to CMAP movements
# (air or international shipping) for Hawaii to non-CMAP movements
# (truck, air or international shipping) for Alaska to CMAP movements
# (truck, air or international shipping) for Alaska to non-CMAP U.S. movements
# (air or international shipping) for Alaska to Hawaii/Foreign movements
# (air or international shipping) for CMAP to Foreign movements
# (air or international shipping) for non-CMAP U.S. to Foreign movements
# (for MINIMUM ALLOWABLE modes): non-CMAP U.S. to non-CMAP U.S. movements
# (for MINIMUM ALLOWABLE DIRECT modes)
# (for MINIMUM ALLOWABLE INDIRECT modes): between CMAP & Rest of U.S. (except Hawaii)/Canada/Mexico
# (for MINIMUM ALLOWABLE INDIRECT modes): between Rest of U.S. (except Hawaii)/Canada/Mexico & CMAP
# (for MINIMUM ALLOWABLE INDIRECT modes): U.S.-Foreign (except Canada/Mexico) shipments

In [ ]:
# Define modepath combinations for 'Bad Data' checks
# non-CMAP U.S. intrazonal movements
# (international water) Data for Canada to CMAP movements
# (CMAP air or CMAP water) Data for Canada to non-CMAP U.S. movements
# (unavailable modes) Data for Canada to Hawaii movements
# (international water) Data for Mexico to CMAP movements
# (CMAP air or CMAP water) Data for Mexico to non-CMAP U.S. movements
# Hawaii to CMAP movements
# Hawaii to non-CMAP movements
# (CMAP water or rail) Data for Alaska to CMAP movements
# (CMAP water, rail or air) Data for Alaska to non-CMAP U.S. movements
# Alaska to Hawaii/Foreign movements
# CMAP to Foreign movements
# non-CMAP U.S. to Foreign movements

In [ ]:
# Define modepath combinations for 'ERROR: No Rail Service' checks
# Canada to CMAP movements
# Canada to non-CMAP U.S. movements
# Mexico to CMAP movements
# Mexico to non-CMAP U.S. movements
# 
# 
# 

In [ ]:
# Define modepath combinations for 'ERROR: There should be NO Rail Service' checks
# Canada & Alaska
# Mexico & Alaska
# non-CMAP U.S. to non-CMAP U.S. movements (x2)

In [ ]:
# Define Modepath Combinations
mp_nonCMAPIntra1 = [3, 4, 13, 14, 31, 32, 39, 46, 47] # Carload direct/indirect: 3, 4; IMX direct/indirect: 13, 14; FTL direct: 31; FTL indirect: 32; LTL direct: 46; LTL indirect: 39; Air: 47
mp_nonCMAPIntra2 = [c for c in allModepaths if c not in mp_nonCMAPIntra1]

mp_CanadaCMAP1 = [1, 2, 31, 46, 47, 48, 50]
mp_CanadaCMAP2 = [51, 52, 53, 54]
mp_CanadaCMAP3 = [range(3, 30)]

mp_CanadaNonCMAP1 = [31, 46, 47, 51, 52, 53, 54]    # Minimum Options available to all zones should be: FTL direct [31], LTL direct [46], air [47], international water [51-54]
mp_CanadaNonCMAP2 = [2, 48, 50]                     # No CMAP water [2 only] or air [48-50] should be available
mp_CanadaNonCMAP3 = [range(3,30)]

# Add logistics node 149 if active
flag48lst = [mp_CanadaCMAP1, mp_CanadaNonCMAP2]
for lst in flag29lst:
    if flag2022!=0:
        lst.append(49)


missingData = {
    'nonCMAPIntra': [cond_nonCMAPIntra, None, mp_nonCMAPIntra1, "non-CMAP U.S. intrazonal movements"],
    'CanadaToCMAP': [cond_CanadaCMAP, None, mp_CanadaCMAP1, "non-CMAP U.S. intrazonal movements"],
    'CanadaToNonCMAP': [cond_CanadaNonCMAP, cond_noHawaii, mp_CanadaNonCMAP1, "Canada to non-CMAP U.S. movements"]
}

badData = {
    'nonCMAPIntra': [cond_nonCMAPIntra, None, mp_nonCMAPIntra2, "non-CMAP U.S. intrazonal movements"],
    'CanadaToCMAP': [cond_CanadaCMAP, None, mp_CanadaCMAP2, "non-CMAP U.S. intrazonal movements"],
    'CanadaToNonCMAP': [cond_CanadaNonCMAP, cond_noHawaii, mp_CanadaNonCMAP2, "(CMAP air or CMAP water) Data for Canada to non-CMAP U.S. movements"]
}

errorData = {
    'CanadaToCMAP': [cond_CanadaCMAP, None, mp_CanadaCMAP3, "non-CMAP U.S. intrazonal movements"],
    'CanadaToNonCMAP1': [cond_CanadaNonCMAP, cond_noAlaska, mp_CanadaNonCMAP3, "Canada to non-CMAP U.S. movements"],
}

for QC in allQC:
    qcDataset = in_modepathcosts.loc[allQC[QC][0]].copy()     # Subset Data 
    if allQC[QC][1] != None:
        qcDataset = qcDataset.loc[allQC[QC][1]].copy()     # Subset Data from condition 2 if applicable
        
    qcModepaths1 = allQC[QC][2]
    qcModepaths2 = allQC[QC][3]
    qcModepaths3 = allQC[QC][4]
    
    # Check for missing data
    for _, row in qcDataset.iterrows(): 
        for path in qcModepaths1:
            if pd.isnull(row[f'time{path}']):
                message = "Missing Time Data for " + allQC[QC][5] + ": 
                print(message)
                print(row[['origin', 'destination', f'time{path}']])
                sys.exit()
            if pd.isnull(row[f'cost{path}']):
                message = "Missing Cost Data for " + allQC[QC][5] + ": 
                print(message)
                print(row[['origin', 'destination', f'cost{path}']])
                sys.exit()

    # Check for data where there should not be data
    for path in qcModepaths2:
        if row[f'time{path}'] > 0:
            message = "Bad Time Data for " + allQC[QC][5] + ": 
            print(message)
            print(row[['origin', 'destination', f'time{path}']])
            sys.exit()
        if row[f'cost{path}'] > 0:
            message = "Bad Cost Data for " + allQC[QC][5] + ": 
            print(message)
            print(row[['origin', 'destination', f'cost{path}']])
            sys.exit()

    # Check for missing Rail service
    for idx in qcModepaths3:
        if ~(row[f'time{idx}'] > 0):
            message = "ERROR: No Rail Service Time found for " + allQC[QC][5] + ": 
            print(message)
            print(row[['origin', 'destination', f'time{path}']])
            sys.exit()
        if ~(row[f'cost{idx}'] > 0):
            message = "ERROR: No Rail Service Cost found for " + allQC[QC][5] + ": 
            print(message)
            print(row[['origin', 'destination', f'cost{path}']])
            sys.exit()      